In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Control**</span>

In [3]:
study = "Control"

### 2. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [4]:
# file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/parameters of interest.xlsx"
file_path = "/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

---

# **Extract <span style="color:blue;">Control**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## **STEP 1**. Filter patients (no pathology, BI-RADS = 1, 2) then merge with patients with DBT available
### (OUTPUT) candidate_cohort

enteredit_findings
* Columns: COMPOSITION_NAME, FINDING_CATEGORY, FINDING_REC, EXAM_COMPLETED_DATE
* Cancer: 
    * FINDING_CATEGORY = 1, 2

pathology
* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Control: 
    * If pathology exists --> exclude

### <span style="color:#FF6347;">**READ**</span> file (EHR: enteredit_findings, pathology)

In [5]:
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

In [6]:
fn = file_name[0]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
birads = pd.read_excel(file_path)

In [7]:
birads.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
0,4330000534,78048053,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-03-28,2
1,4330000534,64787403,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-05-18,2
2,4330000534,63280481,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-06-20,2


In [8]:
fn = file_name[1]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

In [9]:
pathology.head(3)

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330189702,418851,2019-04-26,Malignant,R
1,4330311096,450823,2016-11-10,Malignant,R
2,4330311096,459526,2016-11-28,Malignant,R


In [10]:
fn = file_name[3]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
patient_data_ie = pd.read_excel(file_path)

In [11]:
patient_data_ie.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_NAME,EXAM_CODE,EXAM_NAME,COMPLETED_DATE,ORDER_DATE,EXAM_TYPE,FIRST_MAMMO_IND,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID
0,4330000534,78048053,Magee Womens Imaging South,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,2018-03-28,NaN,Mammography,N,2021-03-09,6909744,8267555
1,4330000534,64787403,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2019-05-18,2019-05-18,Mammography,N,2021-03-09,90405175,8267555
2,4330000534,63280481,Magee Womens Imaging South,BUBRSTCMRT,BREAST ULTRASOUND COMPLETE RIGHT,2019-06-20,NaN,Ultrasound,N,2021-03-09,90988303,8267555


### **0. Merge BI-RADS with patient_data_ie**

In [12]:
# Merge EXAM_NAME into birads from patient_data_ie
exam_name = (
    patient_data_ie[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_NAME']]
    .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])
)

#### <span style="color:blue"> **Study**</span> (SCREEN, DIAG)

In [13]:
study_types = {
    "DIAG":   ["DIAG", "DIAGNOSTIC", "DX"],
    "SCREEN": ["SCREENING", "SCREEN"],
}

In [14]:
column_to_check = 'EXAM_NAME'
type_column = 'Study'

exam_name[column_to_check] = exam_name[column_to_check].astype(str)

for label, terms in study_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = exam_name[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    exam_name.loc[mask, type_column] = label

#### <span style="color:blue">**Series**</span> (DBT)

In [15]:
series_types = {
    "DBT":    ["Breast Tomosynthesis", "TOMOSYNTHESIS", "TOMOSYN", "TOMO"]
}

In [16]:
column_to_check = 'EXAM_NAME'
type_column = 'Series'

exam_name[column_to_check] = exam_name[column_to_check].astype(str)

for label, terms in series_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = exam_name[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    exam_name.loc[mask, type_column] = label

In [17]:
exam_name.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_NAME,Study,Series
0,4330000534,78048053,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT
1,4330000534,64787403,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT
2,4330000534,63280481,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN
3,4330000534,63280358,BREAST ULTRASOUND COMPLETE LEFT,NaN,NaN
4,4330000534,63433254,BREAST IMAGING MISCELLANEOUS FOLLOW UP TECHNIC...,NaN,NaN


In [18]:
birads = birads.merge(exam_name, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='left')

In [19]:
birads.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series
0,4330000534,78048053,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT
1,4330000534,64787403,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-05-18,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT
2,4330000534,63280481,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN
3,4330000534,60228678,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-06-01,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT
4,4330000534,60228678,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-06-01,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT


### **1. Filter out patients with pathology**

In [20]:
birads.drop(columns="FINDING_LOCATION", inplace=True)

In [21]:
# Identify all patient IDs that have pathology records
pathology_pids = pathology['PATIENT_STUDY_ID'].unique()

# Exclude any patient who appears in pathology at all
birads_no_pathology = birads[~birads['PATIENT_STUDY_ID'].isin(pathology_pids)].copy()

print(f"Patients (BI-RADS) : {birads['PATIENT_STUDY_ID'].nunique()}")
print(f"Patients removed (have pathology) : {birads['PATIENT_STUDY_ID'].nunique() - birads_no_pathology['PATIENT_STUDY_ID'].nunique()}")
print(f"Patients (BI-RADS, no pathology) : {birads_no_pathology['PATIENT_STUDY_ID'].nunique()}")

Patients (BI-RADS) : 38339
Patients removed (have pathology) : 362
Patients (BI-RADS, no pathology) : 37977


### **2. Filter BI-RADS findings for Control criteria** (BI-RADS 1 or 2)

In [22]:
# Keep only BI-RADS 1 (Negative) and BI-RADS 2 (Benign) findings
normal_categories = ['1 - Negative', '2 - Benign finding']

birads_control = birads_no_pathology[
    birads_no_pathology['FINDING_CATEGORY'].isin(normal_categories)
].copy()


birads_control.reset_index(inplace=True, drop=True)

print(f"Rows for BIRADS 1 or 2 filter      : {len(birads_control)}")

Rows for BIRADS 1 or 2 filter      : 234737


In [23]:
# Handle bilateral exams: no SIDE + duplicate_count ≥ 2 → expand to L and R
no_side  = birads_control.copy()

bilateral_mask = no_side['duplicate_count'] >= 2
bilateral      = no_side[bilateral_mask].copy()
rest           = no_side[~bilateral_mask].copy()

bilateral_L = bilateral.copy(); bilateral_L['SIDE'] = 'L'
bilateral_R = bilateral.copy(); bilateral_R['SIDE'] = 'R'

birads_control = pd.concat(
    [bilateral_L, bilateral_R, rest], ignore_index=True
)
print(f"birads_control             : {len(birads_control)}")

birads_control             : 415095


In [24]:
birads_control.sort_values(by=["PATIENT_STUDY_ID", "EXAM_COMPLETED_DATE", "ACCESSION_NUMBER"], inplace=True)
birads_control.reset_index(inplace=True, drop=True)

In [25]:
birads_control

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,SIDE
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,L
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,R
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,L
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,R
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,NaN
...,...,...,...,...,...,...,...,...,...,...,...
415090,4339988199,61697072,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-02-21,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R
415091,4339988199,67718778,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L
415092,4339988199,67718778,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R
415093,4339988199,452549960,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2022-03-01,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L


In [26]:
df_step1 = birads_control.copy()

In [27]:
df_step1.shape

(415095, 11)

### <span style="color:#FF6347;">**SAVE**</span> file

In [28]:
# # output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "candidate_cohort" + ".xlsx")
# df_step1.to_excel(output_file, index=False)

## **STEP 2**. <span style="color:blue;">**Control**</span>: Label  "Index" = <span style="color:#8A2BE2;">**INDEX**</span> or <span style="color:#00BFFF;">**INDEX-1**</span>
### (OUTPUT) candidate_cohort

### <span style="color:#FF6347;">**READ**</span> file (candidate_cohort)

In [29]:
# # output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "candidate_cohort" + ".xlsx")
# control_cohort = pd.read_excel(output_file)

In [30]:
control_cohort = df_step1.copy()

In [31]:
control_cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(control_cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")

In [32]:
control_cohort["PATIENT_STUDY_ID"].nunique()

37977

In [33]:
control_cohort.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,SIDE
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,L
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,R
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,L
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,R
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,NaN


## 1. Locate <span style="color:blue;">**Control**</span> year, "Index" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [34]:
threshold_group = pd.Timedelta(days=365/12 * 1.5)  # 1.5-month grouping

In [35]:
cohort = control_cohort.copy()
cohort['Index'] = None
cohort['EXAM_COMPLETED_DATE'] = pd.to_datetime(cohort['EXAM_COMPLETED_DATE'])

In [36]:
# ── STEP 1: Locate INDEX ──────────────────────────────────────────────────────
# 1a. INDEX = latest SCREEN exam with BIRADS 1/2 and image available
# idx_screen = cohort['Study'] == 'SCREEN'
idx_normal = cohort['FINDING_CATEGORY'].isin(['1 - Negative', '2 - Benign finding'])

latest_index = (
    cohort[idx_normal]
    .groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE']
    .max()                    # <── latest qualifying exam
    .reset_index()
    .rename(columns={'EXAM_COMPLETED_DATE': 'INDEX_DATE'})
)

cohort = cohort.merge(latest_index, on='PATIENT_STUDY_ID', how='left')

# 1b. Mark exams within 1.5 months before (and including) the anchor as INDEX
mask_index = (
    cohort['INDEX_DATE'].notna() &
    (cohort['EXAM_COMPLETED_DATE'] <= cohort['INDEX_DATE']) &
    (cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE'] <= threshold_group)
)
cohort.loc[mask_index, 'Index'] = 'INDEX'

print(f"✅ INDEX rows    : {mask_index.sum()}")
print(f"   Patients      : {cohort[mask_index]['PATIENT_STUDY_ID'].nunique()}")

✅ INDEX rows    : 75210
   Patients      : 37977


In [37]:
cohort.head(20)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,SIDE,Index,INDEX_DATE
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,L,None,2022-06-25
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,R,None,2022-06-25
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,L,None,2022-06-25
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,R,None,2022-06-25
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,NaN,None,2022-06-25
5,4330000534,60687102,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-10,1,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT,NaN,None,2022-06-25
6,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,None,2022-06-25
7,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,None,2022-06-25
8,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX,2022-06-25
9,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX,2022-06-25


## 2. Locate <span style="color:blue;">**Control with 1 year follow-up**</span> year, "Index" = <span style="color:#00BFFF;">**INDEX-1**</span>. Adjust <span style="color:pink;"> **screening interval (OR search period)**</span> 

Add threshold labeling for the same index time point (threshold ~1.5 months)

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [38]:
# ── Parameters ────────────────────────────────────────────────────────────────
min_months      = pd.Timedelta(days=0)              # how far back to look (min)
max_months      = pd.Timedelta(days=365/12 * 30)   # how far back to look (max)
threshold_group = pd.Timedelta(days=365/12 * 1.5)  # 1.5-month grouping

In [39]:
cohort['exam_to_index_diff'] = cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE']

mask_window = (
    cohort['exam_to_index_diff'].notna() &
    (cohort['exam_to_index_diff'] > min_months) &
    (cohort['exam_to_index_diff'] <= max_months) &
    (cohort['Index'] != 'INDEX') &
    (cohort['Study'] == 'SCREEN')
)

def assign_index_labels(dates):
    dates_sorted = sorted(set(dates), reverse=True)  # newest → oldest
    labels = {}
    rank = 0
    group_anchor = None
    for d in dates_sorted:
        if group_anchor is None or (group_anchor - d) > threshold_group:
            rank += 1
            group_anchor = d
        labels[d] = f'INDEX-{rank}'
    return labels

window_rows = cohort[mask_window].copy()
date_label_map = {}
for pid, grp in window_rows.groupby('PATIENT_STUDY_ID'):
    mapping = assign_index_labels(grp['EXAM_COMPLETED_DATE'])
    for date, label in mapping.items():
        date_label_map[(pid, date)] = label

cohort.loc[mask_window, 'Index'] = cohort.loc[mask_window].apply(
    lambda row: date_label_map.get(
        (row['PATIENT_STUDY_ID'], row['EXAM_COMPLETED_DATE']),
        row['Index']
    ),
    axis=1
)

print(cohort['Index'].value_counts(dropna=False))

Index
None       233924
INDEX       75210
INDEX-1     64930
INDEX-2     40656
INDEX-3       373
INDEX-4         2
Name: count, dtype: int64


In [40]:
# ── STEP 3: time_to_INDEX ─────────────────────────────────────────────────────
is_prior = cohort['Index'].str.match(r'^INDEX-\d+$', na=False)

cohort['time_to_INDEX_days'] = (
    cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE']
).dt.days

cohort.loc[~is_prior, 'time_to_INDEX_days'] = pd.NA
cohort['time_to_INDEX_months'] = (cohort['time_to_INDEX_days'] / 30.44).round(1)

cohort = cohort.drop(columns=['INDEX_DATE', 'exam_to_index_diff'])

In [41]:
cohort.head(20)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,L,None,NaN,NaN
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,R,None,NaN,NaN
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,L,None,NaN,NaN
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,R,None,NaN,NaN
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,NaN,INDEX-2,754.0,24.8
5,4330000534,60687102,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-10,1,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT,NaN,None,NaN,NaN
6,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX-1,387.0,12.7
7,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX-1,387.0,12.7
8,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX,NaN,NaN
9,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX,NaN,NaN


### <span style="color:#FF6347;">**SAVE**</span> file

In [42]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'candidate_cohort' + ".xlsx")
# cohort.to_excel(output_file, index=False)

In [43]:
cohort["PATIENT_STUDY_ID"].nunique()

37977

In [44]:
df_step2 = cohort.copy()

In [45]:
df_step2.shape

(415095, 14)

## **STEP 3**. Extract <span style="color:blue;">**Normal**</span> cohort only
### (Output) normal, normal_cohort

#### <span style="color:#FF6347;">**READ**</span> file (candidate_cohort)

In [46]:
# # output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'candidate_cohort' + ".xlsx")
# cohort = pd.read_excel(output_file)

In [47]:
cohort = df_step2.copy()

In [48]:
cohort.head(20)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,L,None,NaN,NaN
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,SCREEN,DBT,R,None,NaN,NaN
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,L,None,NaN,NaN
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,BREAST ULTRASOUND COMPLETE RIGHT,NaN,NaN,R,None,NaN,NaN
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,NaN,INDEX-2,754.0,24.8
5,4330000534,60687102,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-10,1,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT,NaN,None,NaN,NaN
6,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX-1,387.0,12.7
7,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX-1,387.0,12.7
8,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX,NaN,NaN
9,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX,NaN,NaN


In [49]:
cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")

### **Criteria**:
#### 1. No pathology ever
* Already met, no need to check
#### 2. At <span style="color:#8A2BE2;">**INDEX**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 3. At <span style="color:#8A2BE2;">**INDEX**</span>, "Study" = "SCREEN"
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT"
#### 5. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 6. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Study" = "SCREEN"

In [50]:
normal_cats = ['1 - Negative', '2 - Benign finding']

recall_cats = [
    '0 - Need additional imaging evaluation',
    '3 - Probably benign - short interval follow-up',
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken',
    '6 - Known biopsy proven malignancy'
]

In [51]:
# Patients with ANY recall finding at INDEX-1 (disqualify, takes priority)
recall_patients = set(cohort.loc[
    ((cohort['Index'] == 'INDEX-1') | (cohort['Index'] == 'INDEX')) &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(recall_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX
normal_patients_at_index = set(cohort.loc[
    (cohort['Index'] == 'INDEX') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX-1
normal_patients_at_index1 = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# # Patients with a normal (BI-RADS 2) finding at INDEX-1
# normal_patients_birads_2_at_index1 = set(cohort.loc[
#     (cohort['Index'] == 'INDEX-1') &
#     (cohort['Series'] == 'DBT') &
#     (cohort['FINDING_CATEGORY'].isin(['2 - Benign finding'])) &
#     (cohort['Study'] == 'SCREEN'),
#     'PATIENT_STUDY_ID'
# ].unique())

# Patients with study=DIAG finding at INDEX-1
patient_diag_at_index1 = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Study'] == 'DIAG'),
    'PATIENT_STUDY_ID'
].unique())

# Patients with study=DIAG finding at INDEX
patient_diag_at_index = set(cohort.loc[
    (cohort['Index'] == 'INDEX') &
    (cohort['Study'] == 'DIAG'),
    'PATIENT_STUDY_ID'
].unique())

normal_patients = normal_patients_at_index & normal_patients_at_index1

# Exclude wins: remove any patient who appeared in recall, also remove any patient who has Study="DIAG" at INDEX-1
qualifying_patients = normal_patients - recall_patients - patient_diag_at_index1 - patient_diag_at_index

print(f"Normal patients     : {len(normal_patients)}")
print(f"Recall patients     : {len(recall_patients)}")
print(f"Overlap removed     : {len(normal_patients & recall_patients)}")
print(f"Qualifying patients : {len(qualifying_patients)}")

normal = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].copy()
print(f"normal rows      : {len(normal)}")
print(f"Index breakdown:\n{normal['Index'].value_counts(dropna=False)}")

Normal patients     : 32295
Recall patients     : 0
Overlap removed     : 0
Qualifying patients : 30047
normal rows      : 341595
Index breakdown:
Index
None       188398
INDEX       59906
INDEX-1     56961
INDEX-2     36213
INDEX-3       117
Name: count, dtype: int64


In [52]:
normal_cohort = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].reset_index(drop=True)
normal_cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True, inplace=True)

normal_cohort["EXAM_COMPLETED_DATE"] = normal_cohort["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

##### <span style="color:#FF6347;">**SAVE**</span> **normal_cohort**

In [53]:
normal_cohort['PATIENT_STUDY_ID'].nunique()

30047

In [ ]:
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'normal_candidate_cohort' + ".xlsx")

normal_cohort.to_excel(output_file, index=False)

##### <span style="color:#FF6347;">**SAVE**</span> **normal** (only retain INDX & INDEX-1)

In [55]:
normal = normal_cohort[(normal_cohort["Index"]=="INDEX") | (normal_cohort["Index"]=="INDEX-1")]

In [56]:
normal['PATIENT_STUDY_ID'].nunique()

30047

In [57]:
normal_cohort.shape, normal.shape

((341595, 14), (116867, 14))

In [60]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'normal' + ".xlsx")
# output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", 'normal_candidate' + ".xlsx")
output_file = os.path.join("../normal_candidate.xlsx")

normal.to_excel(output_file, index=False)

Exception ignored in: <function ZipFile.__del__ at 0x1066de5e0>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/imgprep/lib/python3.8/zipfile.py", line 1821, in __del__
    self.close()
  File "/opt/miniconda3/envs/imgprep/lib/python3.8/zipfile.py", line 1838, in close
    self.fp.seek(self.start_dir)
TimeoutError: [Errno 60] Operation timed out
